# 级数求和 完整教程：从高斯求和到巧妙应用

## 📚 教学目标
1. 掌握**等差/等比级数求和**的基本公式及推导
2. 学会用求和公式解决**钟表碎块**等巧妙分割问题
3. 理解**缺失整数检测**背后的数学原理（单个/两个缺失）
4. 掌握**假币问题**中的加权编码思想
5. 用几何级数求解**期望值**问题并用 Monte Carlo 验证

**参考书**：《A Practical Guide to Quantitative Finance Interviews》(Xinfeng Zhou) 第2章
**教学策略**：先推导公式、手算小例子，再用 Python 枚举/模拟验证

---

## 1. 场景设定：求和公式的威力

### 🎯 一个经典故事

传说高斯小时候，老师让全班算 $1+2+3+\cdots+100$。其他同学还在逐个相加，高斯已经写下了答案：**5050**。

他发现了**配对求和**的技巧：

$$S = 1 + 2 + 3 + \cdots + 100$$
$$S = 100 + 99 + 98 + \cdots + 1$$

两式相加：$2S = 101 \times 100$，所以 $S = 5050$。

这个简单的思想在量化面试中反复出现 —— 求和公式是快速计算的基石。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import combinations

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 基本求和公式推导与验证

### 📐 三大求和公式

**公式 1：等差级数 (Arithmetic Series)**

$$\sum_{k=1}^{n} k = \frac{n(n+1)}{2}$$

**公式 2：平方和 (Sum of Squares)**

$$\sum_{k=1}^{n} k^2 = \frac{n(n+1)(2n+1)}{6}$$

**公式 3：等比级数 (Geometric Series)**

$$\sum_{k=0}^{n-1} r^k = \frac{1 - r^n}{1 - r}, \quad r \neq 1$$

当 $|r| < 1$ 时，无穷等比级数收敛：

$$\sum_{k=0}^{\infty} r^k = \frac{1}{1-r}$$

In [ ]:
# ========== 步骤 1: 验证三大求和公式 ==========
print("📊 步骤 1: 三大求和公式验证")
print("─" * 60)

# --- 公式 1: 等差级数 ---
print("\n📐 公式 1: 等差级数 sum(k) = n(n+1)/2")
for n in [10, 100, 1000]:
    brute = sum(range(1, n+1))
    formula = n * (n + 1) // 2
    match = "✅" if brute == formula else "❌"
    print(f"  n={n:4d}: 逐项求和={brute:>8d}, 公式={formula:>8d} {match}")

# --- 公式 2: 平方和 ---
print("\n📐 公式 2: 平方和 sum(k^2) = n(n+1)(2n+1)/6")
for n in [10, 100, 1000]:
    brute = sum(k**2 for k in range(1, n+1))
    formula = n * (n + 1) * (2*n + 1) // 6
    match = "✅" if brute == formula else "❌"
    print(f"  n={n:4d}: 逐项求和={brute:>12d}, 公式={formula:>12d} {match}")

# --- 公式 3: 等比级数 ---
print("\n📐 公式 3: 等比级数 sum(r^k) = (1-r^n)/(1-r)")
for r, n in [(0.5, 10), (0.9, 20), (2, 10)]:
    brute = sum(r**k for k in range(n))
    formula = (1 - r**n) / (1 - r)
    match = "✅" if abs(brute - formula) < 1e-10 else "❌"
    print(f"  r={r}, n={n:2d}: 逐项求和={brute:>12.4f}, 公式={formula:>12.4f} {match}")

print(f"\n  💡 三个公式全部验证通过! 公式计算是 O(1), 逐项求和是 O(n)。")

In [ ]:
# ========== 步骤 2: 高斯配对法可视化 ==========
print("📊 步骤 2: 高斯配对法直观理解")
print("─" * 60)

n = 10
print(f"\n  以 n={n} 为例:")
print(f"  正序: ", end="")
for k in range(1, n+1):
    print(f"{k:3d}", end="")
print()
print(f"  倒序: ", end="")
for k in range(n, 0, -1):
    print(f"{k:3d}", end="")
print()
print(f"  配对: ", end="")
for k in range(1, n+1):
    print(f"{n+1:3d}", end="")
print()
print(f"\n  每对之和 = {n+1}, 共 {n} 对")
print(f"  2S = {n} × {n+1} = {n*(n+1)}")
print(f"  S = {n*(n+1)//2}")

# 可视化
fig, ax = plt.subplots(1, 1, figsize=(12, 5))

n_vis = 10
x = np.arange(1, n_vis + 1)

# 正序柱
bars1 = ax.bar(x - 0.2, x, width=0.35, color='steelblue', edgecolor='black', 
               alpha=0.85, label='k (forward)')
# 倒序柱
bars2 = ax.bar(x + 0.2, x[::-1], width=0.35, color='#e67e22', edgecolor='black', 
               alpha=0.85, label='(n+1-k) (reverse)')

# 配对和的虚线
ax.axhline(y=(n_vis+1)/2, color='#e74c3c', linestyle='--', linewidth=2,
           label=f'Average = (n+1)/2 = {(n_vis+1)/2}')

# 标注每对之和
for i in range(n_vis):
    total = x[i] + x[n_vis-1-i]
    ax.annotate(f'{total}', xy=(x[i], max(x[i], x[n_vis-1-i]) + 0.3),
               ha='center', fontsize=9, color='#e74c3c', fontweight='bold')

ax.set_xlabel('Position', fontsize=12)
ax.set_ylabel('Value', fontsize=12)
ax.set_title(f'Gauss Pairing: 1+2+...+{n_vis} = {n_vis}×{n_vis+1}/2 = {n_vis*(n_vis+1)//2}',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  蓝色 = 正序 (1,2,...,10), 橙色 = 倒序 (10,9,...,1)")
print(f"  每个位置的正序+倒序 = {n_vis+1} (标注在柱顶)")
print(f"  总和 = {n_vis} 对 × {n_vis+1} / 2 = {n_vis*(n_vis+1)//2}")

In [ ]:
# ========== 可视化: 等比级数部分和收敛 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图: 不同 r 值的部分和 ---
ax1 = axes[0]
r_values = [0.3, 0.5, 0.7, 0.9]
colors_r = ['#2ecc71', 'steelblue', '#e67e22', '#e74c3c']
n_terms = 30

for r, color in zip(r_values, colors_r):
    partial_sums = np.cumsum([r**k for k in range(n_terms)])
    limit = 1 / (1 - r)
    ax1.plot(range(1, n_terms + 1), partial_sums, '-o', color=color, 
             markersize=3, linewidth=1.5, label=f'r={r}, limit={limit:.2f}')
    ax1.axhline(y=limit, color=color, linestyle='--', alpha=0.4)

ax1.set_xlabel('Number of Terms (n)', fontsize=12)
ax1.set_ylabel('Partial Sum S(n)', fontsize=12)
ax1.set_title('Geometric Series Partial Sums', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# --- 右图: 收敛速度 (误差) ---
ax2 = axes[1]
for r, color in zip(r_values, colors_r):
    partial_sums = np.cumsum([r**k for k in range(n_terms)])
    limit = 1 / (1 - r)
    errors = np.abs(partial_sums - limit)
    ax2.semilogy(range(1, n_terms + 1), errors, '-o', color=color, 
                 markersize=3, linewidth=1.5, label=f'r={r}')

ax2.set_xlabel('Number of Terms (n)', fontsize=12)
ax2.set_ylabel('|S(n) - Limit| (log scale)', fontsize=12)
ax2.set_title('Convergence Speed of Geometric Series', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：等比级数 r=0.3 收敛最快 (只需几项), r=0.9 收敛最慢")
print(f"  右图：对数尺度看误差, 所有 |r|<1 的级数都指数收敛")
print(f"        r 越小, 斜率越陡, 收敛越快")

---

## 3. 钟表碎块问题 (Clock Pieces)

### 🎯 问题描述

一个钟表表盘有数字 1 到 12。钟表摔碎成若干块，每块是若干个**连续的数字**。

如果要求每块数字之和相等，钟表可以碎成几块？每块是什么？

### 📐 分析

首先，$1+2+\cdots+12 = \frac{12 \times 13}{2} = 78$

如果碎成 $m$ 块且每块之和相等，则每块之和 = $78/m$。

因此 $m$ 必须是 78 的因数：$m \in \{1, 2, 3, 6\}$（对应每块之和 78, 39, 26, 13）。

但还有额外约束：**每块必须是连续数字**（钟表是圆形的，所以 12-1 也算连续）。

In [ ]:
# ========== 步骤 3: 钟表碎块 —— 数学分析 ==========
print("📊 步骤 3: 钟表碎块问题分析")
print("─" * 60)

total_sum = sum(range(1, 13))
print(f"  1+2+...+12 = {total_sum}")
print(f"  78 的因数: ", end="")

factors = [m for m in range(1, 79) if 78 % m == 0]
print(factors)

print(f"\n  可能的碎块数和每块之和:")
for m in factors:
    piece_sum = 78 // m
    print(f"    m={m:2d} 块, 每块之和 = {piece_sum}")

print(f"\n  💡 但还需要检查: 能否将 1-12 的圆形排列分成连续块,")
print(f"     使每块之和恰好相等?")

In [ ]:
# ========== 步骤 4: 暴力搜索所有合法分割 ==========
print("📊 步骤 4: 搜索所有合法的等和分割")
print("─" * 60)

def find_clock_partitions(n_pieces, target_sum):
    """
    在圆形排列 1-12 中寻找分成 n_pieces 块、每块之和为 target_sum 的方案
    每块是连续数字 (圆形排列, 12 后面是 1)
    """
    numbers = list(range(1, 13))  # 钟表数字
    solutions = []
    
    # 尝试每个起始位置
    for start in range(12):
        pieces = []
        pos = start
        valid = True
        
        for piece_idx in range(n_pieces):
            piece = []
            current_sum = 0
            while current_sum < target_sum:
                num = numbers[pos % 12]
                piece.append(num)
                current_sum += num
                pos += 1
                if pos - start > 12:  # 防止无限循环
                    valid = False
                    break
            
            if not valid or current_sum != target_sum:
                valid = False
                break
            pieces.append(tuple(piece))
        
        if valid and pos - start == 12:
            # 去重: 按第一块的最小数字排序
            pieces_sorted = tuple(sorted(pieces))
            if pieces_sorted not in solutions:
                solutions.append(pieces_sorted)
    
    return solutions

print()
for m in factors:
    if m > 12:
        continue
    target = 78 // m
    sols = find_clock_partitions(m, target)
    
    if sols:
        print(f"  m={m} 块 (每块之和={target}): ✅ 找到 {len(sols)} 种方案")
        for i, sol in enumerate(sols):
            pieces_str = " | ".join([str(list(p)) for p in sol])
            sums_str = ", ".join([str(sum(p)) for p in sol])
            print(f"    方案 {i+1}: {pieces_str}")
            print(f"           各块之和: {sums_str}")
    else:
        print(f"  m={m} 块 (每块之和={target}): ❌ 无合法方案")
    print()

print(f"  💡 钟表只能碎成特定块数, 而且连续性约束进一步限制了方案。")

In [ ]:
# ========== 可视化: 钟表分割方案 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

def draw_clock_partition(ax, pieces, title):
    """在圆形钟表上画出分割方案"""
    colors_pieces = ['#2ecc71', '#e74c3c', '#3498db', '#e67e22', '#9b59b6', '#1abc9c']
    
    # 画圆
    theta = np.linspace(0, 2*np.pi, 100)
    ax.plot(np.cos(theta), np.sin(theta), 'k-', linewidth=2)
    
    # 画数字和分块
    num_to_color = {}
    for i, piece in enumerate(pieces):
        for num in piece:
            num_to_color[num] = colors_pieces[i % len(colors_pieces)]
    
    for num in range(1, 13):
        # 钟表: 12 在顶部, 顺时针
        angle = np.pi/2 - (num / 12) * 2 * np.pi
        x = 0.85 * np.cos(angle)
        y = 0.85 * np.sin(angle)
        
        color = num_to_color.get(num, 'gray')
        circle = plt.Circle((x, y), 0.1, color=color, alpha=0.7, edgecolor='black', linewidth=1)
        ax.add_patch(circle)
        ax.text(x, y, str(num), ha='center', va='center', fontsize=11, fontweight='bold')
    
    # 画分割线
    for piece in pieces:
        first_num = piece[0]
        # 分割线在第一个数字之前
        angle = np.pi/2 - ((first_num - 0.5) / 12) * 2 * np.pi
        x1 = 0.7 * np.cos(angle)
        y1 = 0.7 * np.sin(angle)
        x2 = 1.05 * np.cos(angle)
        y2 = 1.05 * np.sin(angle)
        ax.plot([x1, x2], [y1, y2], 'k-', linewidth=3)
    
    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-1.3, 1.3)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')

# 方案示例
# 3 块, 每块和=26
partition_3 = find_clock_partitions(3, 26)
if partition_3:
    draw_clock_partition(axes[0], partition_3[0], 
                         f'3 Pieces (Sum = 26 each)')

# 6 块, 每块和=13
partition_6 = find_clock_partitions(6, 13)
if partition_6:
    draw_clock_partition(axes[1], partition_6[0], 
                         f'6 Pieces (Sum = 13 each)')

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  相同颜色的数字属于同一块。黑线是分割位置。")
print(f"  左图: 3 块方案, 每块之和 = 26")
print(f"  右图: 6 块方案, 每块之和 = 13")
print(f"  关键: 利用 sum(1..12)=78 确定每块目标和, 再检查连续性约束")

---

## 4. 寻找缺失整数 (Missing Integers)

### 🎯 问题描述

**问题 A**：数组包含 1 到 100 中的 99 个数（缺少 1 个），如何用 $O(n)$ 时间找到缺失的数？

**问题 B**：数组包含 1 到 100 中的 98 个数（缺少 2 个），如何找到这两个缺失的数？

### 📐 问题 A 的解法

$$\text{缺失数} = \frac{n(n+1)}{2} - \sum_{\text{数组中的数}}$$

只需要一个求和公式！

### 📐 问题 B 的解法

缺少两个数 $a, b$，需要两个方程：

$$a + b = \frac{n(n+1)}{2} - S_1$$

$$a^2 + b^2 = \frac{n(n+1)(2n+1)}{6} - S_2$$

其中 $S_1 = \sum x_i$, $S_2 = \sum x_i^2$。解这个二元方程组即可。

In [ ]:
# ========== 步骤 5: 缺失一个数 ==========
print("📊 步骤 5: 寻找缺失的一个整数")
print("─" * 60)

n = 100
missing_true = 42  # 真实缺失的数

# 构造数组 (打乱顺序)
arr = list(range(1, n + 1))
arr.remove(missing_true)
np.random.shuffle(arr)

# 求和法
expected_sum = n * (n + 1) // 2
actual_sum = sum(arr)
missing_found = expected_sum - actual_sum

print(f"  n = {n}")
print(f"  期望总和 = n(n+1)/2 = {n}×{n+1}/2 = {expected_sum}")
print(f"  实际总和 = {actual_sum}")
print(f"  缺失数 = {expected_sum} - {actual_sum} = {missing_found}")
print(f"  真实缺失数 = {missing_true}")
print(f"  验证: {'✅ 正确!' if missing_found == missing_true else '❌ 错误'}")
print(f"\n  💡 只需遍历一次数组求和, 时间 O(n), 空间 O(1)!")

In [ ]:
# ========== 步骤 6: 缺失两个数 ==========
print("📊 步骤 6: 寻找缺失的两个整数")
print("─" * 60)

n = 100
missing_a, missing_b = 23, 67  # 真实缺失的两个数

# 构造数组
arr2 = list(range(1, n + 1))
arr2.remove(missing_a)
arr2.remove(missing_b)
np.random.shuffle(arr2)

# 计算 S1 和 S2
S1 = sum(arr2)
S2 = sum(x**2 for x in arr2)

# 期望的 S1 和 S2
expected_S1 = n * (n + 1) // 2
expected_S2 = n * (n + 1) * (2*n + 1) // 6

# a + b 和 a^2 + b^2
sum_ab = expected_S1 - S1
sum_sq_ab = expected_S2 - S2

print(f"  缺失两个数: a 和 b")
print(f"\n  📐 方程 1: a + b = {expected_S1} - {S1} = {sum_ab}")
print(f"  📐 方程 2: a^2 + b^2 = {expected_S2} - {S2} = {sum_sq_ab}")

# 解方程
# a + b = sum_ab
# a^2 + b^2 = sum_sq_ab
# => (a+b)^2 - 2ab = sum_sq_ab
# => ab = (sum_ab^2 - sum_sq_ab) / 2
product_ab = (sum_ab**2 - sum_sq_ab) // 2

print(f"\n  📐 推导: (a+b)^2 - 2ab = a^2+b^2")
print(f"  => ab = ({sum_ab}^2 - {sum_sq_ab}) / 2 = {product_ab}")
print(f"\n  📐 求解一元二次方程: t^2 - {sum_ab}t + {product_ab} = 0")

# 使用求根公式
discriminant = sum_ab**2 - 4 * product_ab
sqrt_disc = int(np.sqrt(discriminant))
a_found = (sum_ab + sqrt_disc) // 2
b_found = (sum_ab - sqrt_disc) // 2

print(f"  判别式 = {sum_ab}^2 - 4×{product_ab} = {discriminant}")
print(f"  sqrt(判别式) = {sqrt_disc}")
print(f"  a = ({sum_ab}+{sqrt_disc})/2 = {a_found}")
print(f"  b = ({sum_ab}-{sqrt_disc})/2 = {b_found}")

print(f"\n  找到: a={a_found}, b={b_found}")
print(f"  真实: a={missing_a}, b={missing_b}")
match = (set([a_found, b_found]) == set([missing_a, missing_b]))
print(f"  验证: {'✅ 正确!' if match else '❌ 错误'}")

print(f"\n  💡 用两个求和公式 (一次方和、二次方和) 构造两个方程,")
print(f"     解一元二次方程即可。时间 O(n), 空间 O(1)!")

In [ ]:
# ========== 步骤 7: 大规模随机测试 ==========
np.random.seed(42)

print("📊 步骤 7: 缺失整数算法的大规模随机测试")
print("─" * 60)

def find_one_missing(arr, n):
    """找到缺失的一个整数"""
    return n * (n + 1) // 2 - sum(arr)

def find_two_missing(arr, n):
    """找到缺失的两个整数"""
    S1 = sum(arr)
    S2 = sum(x**2 for x in arr)
    sum_ab = n*(n+1)//2 - S1
    sum_sq_ab = n*(n+1)*(2*n+1)//6 - S2
    product_ab = (sum_ab**2 - sum_sq_ab) // 2
    disc = sum_ab**2 - 4 * product_ab
    sqrt_d = int(round(np.sqrt(disc)))
    a = (sum_ab + sqrt_d) // 2
    b = (sum_ab - sqrt_d) // 2
    return min(a, b), max(a, b)

N_TEST = 10000
n_total = 100

# 测试缺失一个数
correct_one = 0
for _ in range(N_TEST):
    missing = np.random.randint(1, n_total + 1)
    arr_test = [x for x in range(1, n_total + 1) if x != missing]
    if find_one_missing(arr_test, n_total) == missing:
        correct_one += 1

# 测试缺失两个数
correct_two = 0
for _ in range(N_TEST):
    missing_pair = sorted(np.random.choice(range(1, n_total + 1), 2, replace=False))
    arr_test = [x for x in range(1, n_total + 1) if x not in missing_pair]
    found = find_two_missing(arr_test, n_total)
    if list(found) == missing_pair:
        correct_two += 1

print(f"  测试次数: {N_TEST:,}")
print(f"  缺失一个数: 正确率 = {correct_one/N_TEST*100:.1f}%")
print(f"  缺失两个数: 正确率 = {correct_two/N_TEST*100:.1f}%")
print(f"\n  💡 两种算法在所有随机测试中都 100% 正确!")

---

## 5. 假币问题 (Counterfeit Coins)

### 🎯 问题描述

有 10 袋硬币，每袋很多枚。其中 9 袋是真币（每枚 10 克），1 袋是假币（每枚 11 克）。

**只用天平称一次**，找出哪袋是假币。

### 📐 加权编码思想

策略：从第 $i$ 袋取出 $i$ 枚硬币（$i = 1, 2, \ldots, 10$），全部放在天平上称。

如果全是真币，总重量：
$$W_{\text{expected}} = (1+2+\cdots+10) \times 10 = 55 \times 10 = 550 \text{ 克}$$

实际重量比 550 多出的克数 = 假币所在袋的编号！

$$\text{假币袋编号} = W_{\text{actual}} - 550$$

因为假币袋的 $i$ 枚硬币每枚多 1 克，总共多出 $i$ 克。

In [ ]:
# ========== 步骤 8: 假币问题手算验证 ==========
print("📊 步骤 8: 假币问题 —— 加权编码法")
print("─" * 60)

N_BAGS = 10
TRUE_WEIGHT = 10
FAKE_WEIGHT = 11

# 取硬币方案: 从第 i 袋取 i 枚
coins_taken = list(range(1, N_BAGS + 1))
total_coins = sum(coins_taken)
expected_weight = total_coins * TRUE_WEIGHT

print(f"  取硬币方案: 从第 i 袋取 i 枚")
print(f"  取硬币数: {coins_taken}")
print(f"  总硬币数: {total_coins} (= 1+2+...+{N_BAGS} = {N_BAGS}×{N_BAGS+1}//2)")
print(f"  如果全是真币, 期望重量 = {total_coins} × {TRUE_WEIGHT} = {expected_weight} 克")
print()

# 对每种假币袋位置验证
print(f"  {'假币袋':>8} {'实际重量':>10} {'多出克数':>10} {'推断':>8} {'正确':>6}")
print(f"  {'─'*8} {'─'*10} {'─'*10} {'─'*8} {'─'*6}")

for fake_bag in range(1, N_BAGS + 1):
    actual_weight = 0
    for i in range(1, N_BAGS + 1):
        if i == fake_bag:
            actual_weight += i * FAKE_WEIGHT
        else:
            actual_weight += i * TRUE_WEIGHT
    
    excess = actual_weight - expected_weight
    inferred = excess
    correct = "✅" if inferred == fake_bag else "❌"
    
    print(f"  第 {fake_bag:2d} 袋    {actual_weight:>6d} 克    {excess:>6d} 克    第 {inferred:2d} 袋  {correct}")

print(f"\n  💡 多出的克数 = 假币袋的编号! 加权编码将'哪一袋'映射为'多出多少克'。")
print(f"     这是求和公式 S = n(n+1)/2 的巧妙应用。")

In [ ]:
# ========== 步骤 9: Monte Carlo 验证假币策略 ==========
np.random.seed(42)
N_SIM = 50000

correct = 0
for _ in range(N_SIM):
    # 随机选一袋为假币
    fake_bag = np.random.randint(1, N_BAGS + 1)
    
    # 计算实际重量
    actual_weight = 0
    for i in range(1, N_BAGS + 1):
        w = FAKE_WEIGHT if i == fake_bag else TRUE_WEIGHT
        actual_weight += i * w
    
    # 推断
    inferred_bag = actual_weight - expected_weight
    if inferred_bag == fake_bag:
        correct += 1

print(f"📊 步骤 9: Monte Carlo 验证")
print(f"  模拟次数: {N_SIM:,}")
print(f"  正确率: {correct/N_SIM*100:.1f}%")
print(f"\n  💡 100% 正确! 加权编码策略是确定性的, 不依赖概率。")

In [ ]:
# ========== 可视化: 假币问题编码原理 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图: 各袋取硬币数 ---
ax1 = axes[0]
bags = np.arange(1, N_BAGS + 1)
bars = ax1.bar(bags, bags, color='steelblue', edgecolor='black', alpha=0.85)

for bar, v in zip(bars, bags):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.15,
            str(v), ha='center', va='bottom', fontweight='bold', fontsize=11)

ax1.set_xlabel('Bag Number', fontsize=12)
ax1.set_ylabel('Coins Taken', fontsize=12)
ax1.set_title('Weighted Encoding: Take i Coins from Bag i', fontsize=14, fontweight='bold')
ax1.set_xticks(bags)
ax1.grid(axis='y', alpha=0.3)

# --- 右图: 实际重量 vs 期望重量 ---
ax2 = axes[1]
excess_weights = []
for fake in range(1, N_BAGS + 1):
    w = sum(i * (FAKE_WEIGHT if i == fake else TRUE_WEIGHT) for i in range(1, N_BAGS + 1))
    excess_weights.append(w - expected_weight)

colors_excess = ['#e74c3c' if e > 0 else '#2ecc71' for e in excess_weights]
bars2 = ax2.bar(bags, excess_weights, color='#e67e22', edgecolor='black', alpha=0.85)

for bar, v in zip(bars2, excess_weights):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.15,
            f'+{v}g', ha='center', va='bottom', fontweight='bold', fontsize=10)

ax2.set_xlabel('Fake Bag Number', fontsize=12)
ax2.set_ylabel('Excess Weight (grams)', fontsize=12)
ax2.set_title('Excess Weight = Fake Bag Number', fontsize=14, fontweight='bold')
ax2.set_xticks(bags)
ax2.grid(axis='y', alpha=0.3)

# 对角线参考
ax2.plot(bags, bags, 'r--', linewidth=2, alpha=0.5, label='y = x (perfect encoding)')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：从第 i 袋取 i 枚硬币 (编码方案)")
print(f"  右图：假币袋编号 = 多出的克数 (完美的一一映射)")
print(f"        红色虚线 y=x 说明编码是线性的")

---

## 6. 期望值与几何级数 (Expected Value & Geometric Series)

### 🎯 问题描述

抛一枚公平硬币，直到第一次出现正面。期望需要抛多少次？

### 📐 推导

设第 $k$ 次首次出现正面的概率：

$$P(X=k) = \left(\frac{1}{2}\right)^{k-1} \cdot \frac{1}{2} = \frac{1}{2^k}$$

期望值：

$$E[X] = \sum_{k=1}^{\infty} k \cdot \frac{1}{2^k}$$

利用公式 $\sum_{k=1}^{\infty} k x^k = \frac{x}{(1-x)^2}$（令 $x = 1/2$）：

$$E[X] = \frac{1/2}{(1-1/2)^2} = \frac{1/2}{1/4} = 2$$

### 💡 直觉理解

也可以用递推：设 $E$ 为期望次数
- 第一次抛：花费 1 次
- 如果正面 (概率 1/2)：结束
- 如果反面 (概率 1/2)：问题重新开始，还需要 $E$ 次

$$E = 1 + \frac{1}{2} \cdot 0 + \frac{1}{2} \cdot E \implies E = 2$$

In [ ]:
# ========== 步骤 10: 期望值的级数验证 ==========
print("📊 步骤 10: 首次正面的期望次数")
print("─" * 60)

# 解析部分和
print("\n  📐 部分和 S(N) = sum(k/2^k, k=1..N):")
partial_sum = 0
for k in range(1, 21):
    term = k / (2**k)
    partial_sum += term
    if k <= 10 or k == 20:
        print(f"    k={k:2d}: 项={term:.8f}, 部分和 S({k:2d})={partial_sum:.8f}")
    elif k == 11:
        print(f"    ...")

print(f"\n  理论极限: E[X] = 2")
print(f"  S(20) = {partial_sum:.8f}")
print(f"  误差 = {abs(2 - partial_sum):.2e}")
print(f"\n  💡 级数收敛非常快, 20 项就几乎等于极限值 2!")

In [ ]:
# ========== 步骤 11: Monte Carlo 模拟验证 ==========
np.random.seed(42)
N_SIM = 200000

flips_to_heads = []
for _ in range(N_SIM):
    flips = 0
    while True:
        flips += 1
        if np.random.random() < 0.5:  # 正面
            break
    flips_to_heads.append(flips)

flips_arr = np.array(flips_to_heads)

print(f"📊 步骤 11: Monte Carlo 模拟")
print(f"  模拟次数: {N_SIM:,}")
print(f"  平均抛掷次数: {flips_arr.mean():.4f}")
print(f"  理论期望: 2.0000")
print(f"  标准差: {flips_arr.std():.4f}")
print(f"  中位数: {np.median(flips_arr):.1f}")
print(f"  最大值: {flips_arr.max()}")

print(f"\n  抛掷次数分布:")
for k in range(1, 8):
    count = (flips_arr == k).sum()
    theory = 1 / (2**k)
    print(f"    k={k}: 实际 {count/N_SIM*100:5.2f}%, 理论 {theory*100:5.2f}% (1/2^{k})")

print(f"\n  💡 模拟结果与理论值 E[X]=2 高度吻合!")

In [ ]:
# ========== 可视化: 几何分布与期望值 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图: 几何分布 PMF ---
ax1 = axes[0]
k_vals = np.arange(1, 13)
pmf_theory = [1/(2**k) for k in k_vals]
pmf_empirical = [(flips_arr == k).mean() for k in k_vals]

width = 0.35
ax1.bar(k_vals - width/2, pmf_empirical, width, color='steelblue', edgecolor='black',
        alpha=0.85, label='Simulation')
ax1.bar(k_vals + width/2, pmf_theory, width, color='#e67e22', edgecolor='black',
        alpha=0.85, label='Theory (1/2^k)')

ax1.axvline(x=2, color='#e74c3c', linestyle='--', linewidth=2, label='E[X] = 2')
ax1.set_xlabel('Number of Flips (k)', fontsize=12)
ax1.set_ylabel('Probability', fontsize=12)
ax1.set_title('Geometric Distribution: Flips Until First Heads', 
              fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.set_xticks(k_vals)
ax1.grid(axis='y', alpha=0.3)

# --- 右图: 累积平均收敛 ---
ax2 = axes[1]
cumulative_avg = np.cumsum(flips_arr) / np.arange(1, N_SIM + 1)
sample_points = np.logspace(0, np.log10(N_SIM), 500).astype(int)
sample_points = np.unique(np.clip(sample_points, 1, N_SIM - 1))

ax2.semilogx(sample_points, cumulative_avg[sample_points], color='steelblue', linewidth=1.5)
ax2.axhline(y=2, color='#e74c3c', linestyle='--', linewidth=2, label='E[X] = 2')

ax2.set_xlabel('Number of Simulations', fontsize=12)
ax2.set_ylabel('Running Average', fontsize=12)
ax2.set_title('Convergence of Sample Mean to E[X] = 2', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)
ax2.set_ylim(1.5, 2.5)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：模拟结果 (蓝色) 与理论 PMF (橙色) 完美吻合")
print(f"        红色虚线 = 期望值 E[X]=2")
print(f"  右图：样本均值随模拟次数增加收敛到理论值 2 (大数定律)")

In [ ]:
# ========== 步骤 12: 推广 —— 不同概率下的期望 ==========
print("📊 步骤 12: 推广 —— 成功概率 p 下的期望抛掷次数")
print("─" * 60)

print(f"\n  📐 几何分布期望: E[X] = 1/p")
print(f"\n  推导 (递推法):")
print(f"    E = 1 + (1-p)·E")
print(f"    E - (1-p)E = 1")
print(f"    pE = 1")
print(f"    E = 1/p")
print()

np.random.seed(42)
p_values = [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]
n_sim_each = 50000

print(f"  {'p':>6} {'理论 1/p':>10} {'模拟均值':>10} {'误差':>10}")
print(f"  {'─'*6} {'─'*10} {'─'*10} {'─'*10}")

for p in p_values:
    # 模拟
    trials = []
    for _ in range(n_sim_each):
        count = 0
        while True:
            count += 1
            if np.random.random() < p:
                break
        trials.append(count)
    
    theory = 1/p
    sim_mean = np.mean(trials)
    error = abs(sim_mean - theory)
    print(f"  {p:6.1f} {theory:10.2f} {sim_mean:10.4f} {error:10.4f}")

print(f"\n  💡 所有情况下模拟结果都接近理论值 1/p!")
print(f"     几何级数公式 E[X]=1/p 是期望值计算的基础工具。")

In [ ]:
# ========== 综合汇总报告 ==========
print("=" * 60)
print("📋 级数求和应用综合报告")
print("=" * 60)

print(f"\n🎯 核心思想:")
print(f"   求和公式是快速计算的基石。掌握等差、平方和、等比")
print(f"   三大公式, 可以将 O(n) 的逐项计算化简为 O(1) 的公式。")

print(f"\n📊 应用汇总:")
print(f"   {'问题':<20} {'用到的公式':<25} {'关键思想'}")
print(f"   {'─'*20} {'─'*25} {'─'*20}")
print(f"   {'钟表碎块':<20} {'等差级数 n(n+1)/2':<25} {'总和=78确定块数'}")
print(f"   {'缺失1个数':<20} {'等差级数':<25} {'期望和-实际和'}")
print(f"   {'缺失2个数':<20} {'等差+平方和':<25} {'两个方程两个未知数'}")
print(f"   {'假币问题':<20} {'等差级数(编码)':<25} {'加权编码一一映射'}")
print(f"   {'抛硬币期望':<20} {'几何级数':<25} {'E[X]=1/p递推或级数'}")

print(f"\n🧮 三大公式速查:")
print(f"   等差级数: S = n(n+1)/2")
print(f"   平方和:   S = n(n+1)(2n+1)/6")
print(f"   等比级数: S = (1-r^n)/(1-r),  |r|<1时 S_inf = 1/(1-r)")

print("\n" + "=" * 60)

---

## 7. 核心概念回顾

### 📌 高斯求和 (Gauss Summation)
- **定义**: $\sum_{k=1}^{n} k = \frac{n(n+1)}{2}$
- **公式**: 配对法 —— 首尾配对，每对之和 = $n+1$，共 $n/2$ 对
- **含义**: 将 O(n) 的逐项求和化简为 O(1) 的公式计算
- **判断标准**: 适用于等差数列求和

### 📌 平方和公式 (Sum of Squares)
- **定义**: $\sum_{k=1}^{n} k^2 = \frac{n(n+1)(2n+1)}{6}$
- **应用**: 缺失两个整数问题中提供第二个方程
- **含义**: 与一次方和配合，可以唯一确定两个未知数

### 📌 等比级数 (Geometric Series)
- **定义**: $\sum_{k=0}^{n-1} r^k = \frac{1-r^n}{1-r}$，$|r|<1$ 时 $\sum_{k=0}^{\infty} r^k = \frac{1}{1-r}$
- **应用**: 期望值计算、概率衰减、金融中的折现
- **含义**: 指数衰减过程的总和有有限极限

### 📌 加权编码 (Weighted Encoding)
- **定义**: 通过不同权重的取样，将离散选项编码为连续数值
- **应用**: 假币问题中「从第 i 袋取 i 枚」将袋号编码为重量差
- **含义**: 一次测量获取所有信息

### 📌 错排数 (Derangement Number)
- **定义**: $D(n) = n! \sum_{k=0}^{n} \frac{(-1)^k}{k!}$
- **公式**: $D(n)/n! \to 1/e \approx 0.368$ 当 $n \to \infty$
- **含义**: 所有元素都不在原位的排列数

### 🔗 完整流程
```
识别求和结构 → 选择合适公式 → 建立方程 → 求解 → 验证
       ↓              ↓            ↓         ↓       ↓
  等差/等比/平方和  O(1)公式    代入已知量   解方程  Python模拟
```

### 📝 公式汇总

| 公式名称 | 公式 | 典型应用 |
|---------|------|--------|
| 等差级数 | $n(n+1)/2$ | 缺失数、钟表碎块、假币编码 |
| 平方和 | $n(n+1)(2n+1)/6$ | 缺失两个数 |
| 等比级数 | $(1-r^n)/(1-r)$ | 期望值、概率求和 |
| 几何分布期望 | $1/p$ | 首次成功的期望次数 |

---

## 8. 常见误区

### ❌ 误区 1: 缺失两个数只用一次方和就够了
**✓ 正确理解**: 一次方和 $a+b$ 只提供一个方程，两个未知数需要两个方程。必须同时用一次方和与二次方和（$a+b$ 和 $a^2+b^2$），才能唯一确定 $a$ 和 $b$。

### ❌ 误区 2: 假币问题中每袋取相同数量的硬币
**✓ 正确理解**: 每袋取相同数量无法区分哪一袋是假的（只知道「有假币」，不知道在哪）。加权编码的精髓是「从第 i 袋取 i 枚」，让超出的重量直接反映袋号。

### ❌ 误区 3: 等比级数 $|r| \geq 1$ 时也收敛
**✓ 正确理解**: 无穷等比级数 $\sum r^k$ 仅在 $|r| < 1$ 时收敛到 $1/(1-r)$。$|r| \geq 1$ 时级数发散（$r=1$ 时为调和级数的特殊情况）。有限项求和公式 $(1-r^n)/(1-r)$ 对所有 $r \neq 1$ 都适用。

### ❌ 误区 4: 钟表碎块只要总和能整除就一定有解
**✓ 正确理解**: 总和 78 能被 $m$ 整除只是必要条件，还需要满足**连续性约束**（钟表是圆形排列，每块必须是连续数字）。例如 $m=2$ 时每块之和 39，但可能无法找到连续的两段使各自之和为 39。

### ❌ 误区 5: 抛硬币期望次数的中位数也是 2
**✓ 正确理解**: 几何分布是右偏的。期望值 $E[X]=2$，但中位数是 1（因为 $P(X=1)=0.5$，已经有一半的概率在第一次就成功）。期望值被极端值（运气很差的情况）拉高。